# 02 — Building the AI data analyst

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — AI Data Analyst |
| **Level** | Advanced |
| **Time** | ~40 min |
| **Prerequisites** | [01_architecture.ipynb](./01_architecture.ipynb) |
| **Open in Colab** | [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai2026/castalia/blob/main/labs/06_ai_data_analyst/02_build.ipynb) |

**What you'll build:**
- Realistic e-commerce SQLite database with 1000+ rows
- Schema documentation RAG system
- Full text-to-SQL agent with self-correction
- Insight narration engine
- Auto-chart generation
- End-to-end demo with 10 business questions

## 1 · Setup

In [ ]:
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "sentence-transformers>=2.2.0"

import os
import json
import sqlite3
import random
import hashlib
from datetime import datetime, timedelta
from typing import Optional
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"
random.seed(42)
np.random.seed(42)

print(f"✅ Setup complete — model: {MODEL}")

## 2 · Sample e-commerce database

We create a realistic e-commerce database with five tables and 1000+ rows
of synthetic but plausible data. This is our test bed for the entire lab.

In [ ]:
DB_PATH = "ecommerce_analyst.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript("""
    DROP TABLE IF EXISTS order_items;
    DROP TABLE IF EXISTS orders;
    DROP TABLE IF EXISTS products;
    DROP TABLE IF EXISTS categories;
    DROP TABLE IF EXISTS customers;

    CREATE TABLE categories (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL UNIQUE
    );

    CREATE TABLE products (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        category_id INTEGER NOT NULL REFERENCES categories(id),
        base_price REAL NOT NULL
    );

    CREATE TABLE customers (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT NOT NULL UNIQUE,
        region TEXT NOT NULL CHECK(region IN ('NA','EMEA','APAC','LATAM')),
        segment TEXT NOT NULL CHECK(segment IN ('Enterprise','SMB','Consumer')),
        created_at DATE NOT NULL
    );

    CREATE TABLE orders (
        id INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL REFERENCES customers(id),
        order_date DATE NOT NULL,
        status TEXT NOT NULL CHECK(status IN ('pending','shipped','delivered','returned'))
    );

    CREATE TABLE order_items (
        id INTEGER PRIMARY KEY,
        order_id INTEGER NOT NULL REFERENCES orders(id),
        product_id INTEGER NOT NULL REFERENCES products(id),
        quantity INTEGER NOT NULL CHECK(quantity > 0),
        unit_price REAL NOT NULL CHECK(unit_price > 0)
    );
""")

print("✅ Schema created")
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"  Tables: {tables['name'].tolist()}")

In [ ]:
# Populate categories and products

CATEGORIES = ["Electronics", "Clothing", "Books", "Home & Garden", "Sports", "Toys", "Food & Drink"]

PRODUCTS_BY_CATEGORY: dict[str, list[tuple[str, float]]] = {
    "Electronics": [
        ("Laptop Pro 15", 1299.99), ("Wireless Mouse", 29.99), ("USB-C Hub", 49.99),
        ("Bluetooth Speaker", 79.99), ("Smartphone X", 899.99), ("Tablet Air", 599.99),
        ("Noise-Cancelling Headphones", 249.99), ("Webcam HD", 69.99)
    ],
    "Clothing": [
        ("Cotton T-Shirt", 24.99), ("Denim Jacket", 89.99), ("Running Shoes", 129.99),
        ("Wool Sweater", 69.99), ("Linen Pants", 59.99), ("Baseball Cap", 19.99)
    ],
    "Books": [
        ("Python Mastery", 49.99), ("Data Science Handbook", 39.99),
        ("ML Engineering", 54.99), ("SQL Deep Dive", 34.99), ("AI Strategy", 29.99)
    ],
    "Home & Garden": [
        ("LED Desk Lamp", 44.99), ("Plant Pot Set", 34.99), ("Smart Thermostat", 149.99),
        ("Coffee Maker Pro", 89.99), ("Robot Vacuum", 299.99)
    ],
    "Sports": [
        ("Yoga Mat", 29.99), ("Dumbbell Set", 79.99), ("Fitness Tracker", 149.99),
        ("Tennis Racket", 99.99), ("Cycling Helmet", 59.99)
    ],
    "Toys": [
        ("Building Blocks Set", 39.99), ("Remote Control Car", 49.99),
        ("Board Game Collection", 34.99), ("Science Kit", 29.99)
    ],
    "Food & Drink": [
        ("Premium Coffee Beans", 18.99), ("Organic Tea Set", 24.99),
        ("Gourmet Chocolate Box", 29.99), ("Artisan Olive Oil", 15.99)
    ]
}

# Insert categories
for i, cat in enumerate(CATEGORIES, 1):
    cursor.execute("INSERT INTO categories VALUES (?, ?)", (i, cat))

# Insert products
product_id = 1
for cat_idx, (cat, products) in enumerate(PRODUCTS_BY_CATEGORY.items(), 1):
    for name, price in products:
        cursor.execute("INSERT INTO products VALUES (?, ?, ?, ?)",
                       (product_id, name, cat_idx, price))
        product_id += 1

conn.commit()
total_products = cursor.execute("SELECT COUNT(*) FROM products").fetchone()[0]
print(f"✅ Inserted {len(CATEGORIES)} categories, {total_products} products")

In [ ]:
# Populate customers (200 customers)

FIRST_NAMES = ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace", "Hank",
               "Ivy", "Jack", "Karen", "Leo", "Mia", "Nick", "Olivia", "Pete",
               "Quinn", "Rosa", "Sam", "Tina", "Uma", "Victor", "Wendy", "Xander",
               "Yara", "Zane", "Aria", "Blake", "Cleo", "Dean"]
LAST_NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller",
              "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Wilson",
              "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee"]
REGIONS = ["NA", "EMEA", "APAC", "LATAM"]
REGION_WEIGHTS = [0.4, 0.3, 0.2, 0.1]
SEGMENTS = ["Enterprise", "SMB", "Consumer"]
SEGMENT_WEIGHTS = [0.2, 0.35, 0.45]

for cust_id in range(1, 201):
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)
    name = f"{first} {last}"
    email_hash = hashlib.md5(f"{cust_id}".encode()).hexdigest()[:8]
    email = f"{first.lower()}.{last.lower()}.{email_hash}@example.com"
    region = random.choices(REGIONS, weights=REGION_WEIGHTS, k=1)[0]
    segment = random.choices(SEGMENTS, weights=SEGMENT_WEIGHTS, k=1)[0]
    created = datetime(2022, 1, 1) + timedelta(days=random.randint(0, 730))

    cursor.execute("INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?)",
                   (cust_id, name, email, region, segment, created.strftime("%Y-%m-%d")))

conn.commit()
count = cursor.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
print(f"✅ Inserted {count} customers")

# Show distribution
region_dist = pd.read_sql_query(
    "SELECT region, COUNT(*) as count FROM customers GROUP BY region ORDER BY count DESC", conn)
print(f"  Region distribution:\n{region_dist.to_string(index=False)}")

In [ ]:
# Populate orders and order_items (800+ orders, 1500+ items)

STATUSES = ["pending", "shipped", "delivered", "returned"]
STATUS_WEIGHTS = [0.05, 0.10, 0.80, 0.05]

order_id = 1
item_id = 1
all_product_ids = [r[0] for r in cursor.execute("SELECT id FROM products").fetchall()]
all_product_prices = {r[0]: r[1] for r in cursor.execute("SELECT id, base_price FROM products").fetchall()}

for month_offset in range(24):  # 24 months of data
    base_date = datetime(2023, 1, 1) + timedelta(days=month_offset * 30)
    # Seasonal variation: more orders in Q4
    month_num = base_date.month
    seasonal_factor = 1.3 if month_num >= 10 else (0.9 if month_num <= 2 else 1.0)
    n_orders = int(random.gauss(35, 8) * seasonal_factor)

    for _ in range(max(10, n_orders)):
        cust_id = random.randint(1, 200)
        order_date = base_date + timedelta(days=random.randint(0, 29))
        status = random.choices(STATUSES, weights=STATUS_WEIGHTS, k=1)[0]

        cursor.execute("INSERT INTO orders VALUES (?, ?, ?, ?)",
                       (order_id, cust_id, order_date.strftime("%Y-%m-%d"), status))

        # 1-4 items per order
        n_items = random.choices([1, 2, 3, 4], weights=[0.4, 0.35, 0.2, 0.05], k=1)[0]
        chosen_products = random.sample(all_product_ids, min(n_items, len(all_product_ids)))

        for prod_id in chosen_products:
            qty = random.choices([1, 2, 3, 5], weights=[0.6, 0.25, 0.1, 0.05], k=1)[0]
            price = all_product_prices[prod_id] * random.uniform(0.85, 1.15)  # price variation
            cursor.execute("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",
                           (item_id, order_id, prod_id, qty, round(price, 2)))
            item_id += 1

        order_id += 1

conn.commit()
order_count = cursor.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
item_count = cursor.execute("SELECT COUNT(*) FROM order_items").fetchone()[0]
print(f"✅ Inserted {order_count} orders, {item_count} order items")

# Quick validation
sample = pd.read_sql_query(
    "SELECT o.id, c.name, o.order_date, COUNT(oi.id) as items, "
    "SUM(oi.quantity * oi.unit_price) as total "
    "FROM orders o JOIN customers c ON o.customer_id = c.id "
    "JOIN order_items oi ON o.id = oi.order_id "
    "GROUP BY o.id LIMIT 5", conn)
print(f"\nSample orders:\n{sample.to_string(index=False)}")

## 3 · Schema documentation RAG

We describe every table, column, and relationship with business context,
then build a simple retriever that surfaces relevant schema for each question.

In [ ]:
SCHEMA_DOCS: dict[str, dict] = {
    "tables": {
        "customers": {
            "description": "Registered customers with profile, region, and segment data",
            "columns": {
                "id": {"type": "INTEGER", "desc": "Auto-increment primary key"},
                "name": {"type": "TEXT", "desc": "Full customer name"},
                "email": {"type": "TEXT", "desc": "Unique email address"},
                "region": {"type": "TEXT", "desc": "Geographic: NA, EMEA, APAC, LATAM"},
                "segment": {"type": "TEXT", "desc": "Business: Enterprise, SMB, Consumer"},
                "created_at": {"type": "DATE", "desc": "Account registration date"}
            },
            "keywords": ["customer", "buyer", "client", "account", "region", "segment",
                         "enterprise", "smb", "consumer", "signup", "registered"]
        },
        "orders": {
            "description": "Purchase orders placed by customers",
            "columns": {
                "id": {"type": "INTEGER", "desc": "Primary key"},
                "customer_id": {"type": "INTEGER", "desc": "FK → customers.id"},
                "order_date": {"type": "DATE", "desc": "Date placed (YYYY-MM-DD)"},
                "status": {"type": "TEXT", "desc": "pending | shipped | delivered | returned"}
            },
            "keywords": ["order", "purchase", "buy", "transaction", "sale", "date",
                         "month", "quarter", "year", "trend", "status", "delivered", "returned"]
        },
        "order_items": {
            "description": "Line items within each order (links orders to products)",
            "columns": {
                "id": {"type": "INTEGER", "desc": "Primary key"},
                "order_id": {"type": "INTEGER", "desc": "FK → orders.id"},
                "product_id": {"type": "INTEGER", "desc": "FK → products.id"},
                "quantity": {"type": "INTEGER", "desc": "Units purchased"},
                "unit_price": {"type": "REAL", "desc": "Price per unit at time of order"}
            },
            "keywords": ["item", "quantity", "price", "revenue", "amount", "total",
                         "spend", "cost", "sales", "value", "average"]
        },
        "products": {
            "description": "Product catalog with pricing",
            "columns": {
                "id": {"type": "INTEGER", "desc": "Primary key"},
                "name": {"type": "TEXT", "desc": "Product display name"},
                "category_id": {"type": "INTEGER", "desc": "FK → categories.id"},
                "base_price": {"type": "REAL", "desc": "Standard retail price"}
            },
            "keywords": ["product", "item", "goods", "merchandise", "sku", "name"]
        },
        "categories": {
            "description": "Product category classification",
            "columns": {
                "id": {"type": "INTEGER", "desc": "Primary key"},
                "name": {"type": "TEXT", "desc": "Category name (e.g., Electronics)"}
            },
            "keywords": ["category", "type", "group", "department", "classification"]
        }
    },
    "relationships": [
        {"from": "orders.customer_id", "to": "customers.id", "desc": "Each order belongs to one customer"},
        {"from": "order_items.order_id", "to": "orders.id", "desc": "Each line item belongs to one order"},
        {"from": "order_items.product_id", "to": "products.id", "desc": "Each line item references one product"},
        {"from": "products.category_id", "to": "categories.id", "desc": "Each product is in one category"}
    ],
    "glossary": {
        "revenue": "SUM(order_items.quantity * order_items.unit_price)",
        "AOV": "Average Order Value = total revenue / number of distinct orders",
        "active customer": "Customer with ≥1 order in the last 90 days",
        "churn": "Customer with no orders in the last 180 days",
        "repeat customer": "Customer with ≥2 lifetime orders",
        "basket size": "Average number of items per order",
        "conversion rate": "Orders / unique customers in a period"
    },
    "example_queries": [
        {"q": "Total revenue", "sql": "SELECT SUM(quantity * unit_price) AS total_revenue FROM order_items;"},
        {"q": "Revenue by region", "sql": "SELECT c.region, SUM(oi.quantity * oi.unit_price) AS revenue FROM customers c JOIN orders o ON c.id = o.customer_id JOIN order_items oi ON o.id = oi.order_id GROUP BY c.region;"},
        {"q": "Top 5 products by units sold", "sql": "SELECT p.name, SUM(oi.quantity) AS units FROM products p JOIN order_items oi ON p.id = oi.product_id GROUP BY p.name ORDER BY units DESC LIMIT 5;"},
        {"q": "Monthly order count", "sql": "SELECT strftime('%Y-%m', order_date) AS month, COUNT(*) AS orders FROM orders GROUP BY month ORDER BY month;"},
        {"q": "Customer order count", "sql": "SELECT c.name, COUNT(o.id) AS orders FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.name ORDER BY orders DESC LIMIT 10;"}
    ]
}

print(f"Schema docs loaded:")
print(f"  Tables: {list(SCHEMA_DOCS['tables'].keys())}")
print(f"  Relationships: {len(SCHEMA_DOCS['relationships'])}")
print(f"  Glossary terms: {len(SCHEMA_DOCS['glossary'])}")
print(f"  Example queries: {len(SCHEMA_DOCS['example_queries'])}")

In [ ]:
def retrieve_context(question: str, schema: dict) -> dict[str, object]:
    """
    Retrieve relevant schema context for a question using keyword matching.

    Args:
        question: Natural-language question

    Returns:
        Dict with tables, columns, relationships, glossary, examples
    """
    q_lower = question.lower()
    q_words = set(q_lower.split())

    # Score each table by keyword overlap
    table_scores: dict[str, float] = {}
    for tname, tinfo in schema["tables"].items():
        score = sum(2.0 for kw in tinfo["keywords"] if kw in q_lower)
        score += sum(0.5 for col in tinfo["columns"] if col in q_lower)
        table_scores[tname] = score

    # Select tables with score > 0, or all if none match
    relevant = sorted(
        [t for t, s in table_scores.items() if s > 0],
        key=lambda t: table_scores[t], reverse=True
    ) or list(schema["tables"].keys())

    # Add join-neighbor tables
    expanded = set(relevant)
    for rel in schema["relationships"]:
        from_table = rel["from"].split(".")[0]
        to_table = rel["to"].split(".")[0]
        if from_table in expanded or to_table in expanded:
            expanded.add(from_table)
            expanded.add(to_table)

    # Build context string
    context_parts: list[str] = ["DATABASE SCHEMA:"]
    for tname in expanded:
        tinfo = schema["tables"][tname]
        cols = ", ".join(f"{c} ({ci['type']})" for c, ci in tinfo["columns"].items())
        context_parts.append(f"\nTABLE {tname}: {tinfo['description']}")
        context_parts.append(f"  Columns: {cols}")

    context_parts.append("\nRELATIONSHIPS:")
    for rel in schema["relationships"]:
        context_parts.append(f"  {rel['from']} → {rel['to']}: {rel['desc']}")

    # Match glossary
    matched_glossary = [
        f"  {term}: {defn}"
        for term, defn in schema["glossary"].items()
        if term.lower() in q_lower or any(w in term.lower() for w in q_words if len(w) > 3)
    ]
    if matched_glossary:
        context_parts.append("\nBUSINESS DEFINITIONS:")
        context_parts.extend(matched_glossary)

    # Match examples
    matched_examples = [
        f"  Q: {ex['q']}\n  SQL: {ex['sql']}"
        for ex in schema["example_queries"]
        if any(w in ex["q"].lower() for w in q_words if len(w) > 3)
    ]
    if matched_examples:
        context_parts.append("\nEXAMPLE QUERIES:")
        context_parts.extend(matched_examples)

    return {
        "tables": list(expanded),
        "context_str": "\n".join(context_parts),
        "n_glossary": len(matched_glossary),
        "n_examples": len(matched_examples)
    }


# Test retrieval
for q in ["What is total revenue by category?", "How many customers per region?",
          "Show monthly order trends", "What is the average order value?"]:
    ctx = retrieve_context(q, SCHEMA_DOCS)
    print(f"Q: {q}")
    print(f"  Tables: {ctx['tables']} | Glossary: {ctx['n_glossary']} | Examples: {ctx['n_examples']}")

## 4 · Text-to-SQL agent

The full agent pipeline:
1. Retrieve schema context
2. Disambiguate the question
3. Generate SQL with LLM
4. Execute on SQLite
5. Validate results
6. Self-correct on failure (up to 3 attempts)
7. Narrate the insight

In [ ]:
@dataclass
class AnalystResult:
    """Complete result from the AI data analyst."""
    question: str
    assumptions: list[str]
    sql: str
    data: Optional[pd.DataFrame] = None
    narration: str = ""
    chart_type: str = "table"
    success: bool = False
    attempts: int = 1
    error: Optional[str] = None


def disambiguate(question: str) -> tuple[str, list[str]]:
    """Make implicit assumptions explicit."""
    q_lower = question.lower()
    assumptions: list[str] = []

    ambiguous = {
        "top": "by revenue (descending)",
        "best": "highest revenue",
        "popular": "most units sold",
        "recent": "last 90 days",
        "active": "ordered in last 90 days",
        "growth": "revenue growth month-over-month",
    }

    for term, default in ambiguous.items():
        if term in q_lower:
            assumptions.append(f'"{term}" → {default}')

    time_words = {"today", "yesterday", "week", "month", "quarter", "year",
                  "2023", "2024", "last", "this"}
    if not any(tw in q_lower for tw in time_words):
        assumptions.append("Time scope: all available data")

    import re
    if ("top" in q_lower or "best" in q_lower) and not re.search(r"\b\d+\b", question):
        assumptions.append("Limit: top 10 results")

    return question, assumptions


def generate_sql(question: str, context: str,
                 assumptions: list[str],
                 error_ctx: Optional[str] = None) -> str:
    """Generate SQL from question + schema context via LLM."""
    prompt_parts = [context, f"\nQUESTION: {question}"]

    if assumptions:
        prompt_parts.append("\nASSUMPTIONS:\n" + "\n".join(f"  • {a}" for a in assumptions))

    if error_ctx:
        prompt_parts.append(f"\nPREVIOUS ERROR (fix this):\n{error_ctx}")

    prompt_parts.append("\nGenerate a single SQLite SQL query. Return ONLY the SQL.")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content":
             "You are a SQL expert. Generate valid SQLite SQL. Return ONLY the query, no markdown."},
            {"role": "user", "content": "\n".join(prompt_parts)}
        ],
        temperature=0.0,
        max_tokens=600
    )

    sql = response.choices[0].message.content.strip()
    return sql.replace("```sql", "").replace("```", "").strip()


def execute_and_validate(sql: str, conn: sqlite3.Connection) -> tuple[bool, Optional[pd.DataFrame], Optional[str]]:
    """Execute SQL and validate the result."""
    try:
        df = pd.read_sql_query(sql, conn)
    except Exception as e:
        return False, None, f"Execution error: {e}"

    if df.empty:
        return False, df, "Empty result set"
    if len(df) > 10_000:
        return False, df, f"Suspicious row count: {len(df)} (possible cartesian product)"

    null_cols = [c for c in df.columns if df[c].isna().all()]
    if null_cols:
        return False, df, f"All-NULL columns: {null_cols}"

    return True, df, None


print("✅ Agent components defined")

In [ ]:
def ask_analyst(question: str, conn: sqlite3.Connection,
                max_attempts: int = 3) -> AnalystResult:
    """
    Full AI data analyst pipeline: question → SQL → results → narration.

    Args:
        question: Natural-language business question
        conn: SQLite database connection
        max_attempts: Max self-correction attempts

    Returns:
        AnalystResult with SQL, data, narration, and metadata
    """
    # Step 1: Retrieve schema context
    ctx = retrieve_context(question, SCHEMA_DOCS)

    # Step 2: Disambiguate
    _, assumptions = disambiguate(question)

    # Step 3-6: Generate → Execute → Validate → Self-correct
    error_ctx: Optional[str] = None

    for attempt in range(1, max_attempts + 1):
        sql = generate_sql(question, ctx["context_str"], assumptions, error_ctx)
        valid, data, error = execute_and_validate(sql, conn)

        if valid and data is not None:
            return AnalystResult(
                question=question, assumptions=assumptions,
                sql=sql, data=data, success=True, attempts=attempt
            )

        error_ctx = f"SQL: {sql}\nError: {error}"

    return AnalystResult(
        question=question, assumptions=assumptions,
        sql=sql, success=False, attempts=max_attempts,
        error=error
    )


# Quick test
result = ask_analyst("How many customers do we have?", conn)
print(f"Q: {result.question}")
print(f"SQL: {result.sql}")
print(f"Success: {result.success} (attempts: {result.attempts})")
if result.data is not None:
    print(f"Result:\n{result.data.to_string(index=False)}")

## 5 · Insight narration

Transforms raw query results + the original question into
plain-English insights with findings, trends, and recommendations.

In [ ]:
def narrate_results(question: str, sql: str,
                    data: pd.DataFrame,
                    assumptions: list[str]) -> str:
    """
    Generate a plain-English narrative from query results.

    Args:
        question: Original user question
        sql: The executed SQL query
        data: Query result as DataFrame
        assumptions: Assumptions made during disambiguation

    Returns:
        Narrative string with findings and recommendations
    """
    # Truncate large results for the prompt
    data_preview = data.head(20).to_string(index=False)

    prompt = f"""You are a business analyst narrating query results.

QUESTION: {question}
SQL: {sql}
ASSUMPTIONS: {'; '.join(assumptions) if assumptions else 'None'}

RESULTS:
{data_preview}

Total rows: {len(data)}

Write a concise (3-5 sentence) business insight covering:
1. Direct answer to the question
2. Key finding or notable pattern
3. One actionable recommendation

Use specific numbers from the data. Be precise and professional."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior business analyst. Be concise, data-driven, and actionable."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


# Test narration on several results
test_questions = [
    "What is the total revenue by region?",
    "Show me the top 5 products by revenue",
    "What is the monthly order count trend?",
    "How many customers per segment?",
    "What is the average order value by category?"
]

for q in test_questions:
    result = ask_analyst(q, conn)
    if result.success and result.data is not None:
        narration = narrate_results(q, result.sql, result.data, result.assumptions)
        result.narration = narration
        print(f"Q: {q}")
        print(f"  📊 {narration[:150]}...")
        print()

## 6 · Auto-chart generation

Based on the query type and result shape, we automatically select
and generate the most appropriate visualization.

In [ ]:
def select_chart_type(question: str, data: pd.DataFrame) -> str:
    """
    Select the best chart type based on question and data shape.

    Args:
        question: Original question
        data: Query result DataFrame

    Returns:
        Chart type: 'bar', 'line', 'pie', 'hbar', or 'table'
    """
    q_lower = question.lower()
    n_rows, n_cols = data.shape

    # Time series → line chart
    time_words = ["trend", "monthly", "over time", "by month", "by quarter", "by year"]
    if any(tw in q_lower for tw in time_words) and n_rows >= 3:
        return "line"

    # Distribution / composition → pie (if few categories)
    if n_rows <= 6 and n_cols == 2:
        numeric_cols = data.select_dtypes(include=["number"]).columns
        if len(numeric_cols) == 1:
            return "pie"

    # Ranking → horizontal bar
    rank_words = ["top", "best", "most", "highest", "largest"]
    if any(rw in q_lower for rw in rank_words):
        return "hbar"

    # Comparison → bar chart
    if n_rows <= 20 and n_cols >= 2:
        return "bar"

    return "table"


def generate_chart(question: str, data: pd.DataFrame,
                   chart_type: Optional[str] = None) -> str:
    """
    Generate an appropriate chart for the query results.

    Args:
        question: Original question (for title)
        data: Query result DataFrame
        chart_type: Override chart type, or auto-detect

    Returns:
        Chart type used
    """
    if chart_type is None:
        chart_type = select_chart_type(question, data)

    fig, ax = plt.subplots(figsize=(10, 5))
    label_col = data.columns[0]
    value_col = data.columns[-1] if len(data.columns) > 1 else data.columns[0]

    if chart_type == "line":
        ax.plot(data[label_col], data[value_col], marker="o", linewidth=2)
        ax.set_xlabel(label_col)
        ax.set_ylabel(value_col)
        plt.xticks(rotation=45, ha="right")

    elif chart_type == "bar":
        colors = plt.cm.Set2(range(len(data)))
        ax.bar(data[label_col], data[value_col], color=colors)
        ax.set_xlabel(label_col)
        ax.set_ylabel(value_col)
        plt.xticks(rotation=45, ha="right")

    elif chart_type == "hbar":
        ax.barh(data[label_col], data[value_col], color=plt.cm.Set2(range(len(data))))
        ax.set_xlabel(value_col)
        ax.invert_yaxis()

    elif chart_type == "pie":
        ax.pie(data[value_col], labels=data[label_col],
               autopct="%1.1f%%", startangle=90, colors=plt.cm.Set3.colors)
        ax.axis("equal")

    else:  # table
        ax.axis("off")
        tbl = ax.table(cellText=data.values, colLabels=data.columns,
                       loc="center", cellLoc="center")
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)

    ax.set_title(question, fontsize=12, pad=15)
    plt.tight_layout()
    plt.show()

    return chart_type


# Test chart type selection
chart_tests = [
    ("What is the monthly revenue trend?", "line"),
    ("Show top 5 products by revenue", "hbar"),
    ("What is the revenue breakdown by region?", "pie"),
    ("Compare order counts by category", "bar"),
]

for q, expected in chart_tests:
    result = ask_analyst(q, conn)
    if result.success and result.data is not None:
        actual = select_chart_type(q, result.data)
        match = "✅" if actual == expected else "⚠️"
        print(f"{match} Q: {q} → {actual} (expected: {expected})")

## 7 · End-to-end demo

Ten diverse business questions through the full pipeline:
question → SQL → results → narration → chart.

In [ ]:
DEMO_QUESTIONS: list[dict[str, str]] = [
    {"q": "How many customers do we have in each region?", "difficulty": "easy"},
    {"q": "What is the total revenue?", "difficulty": "easy"},
    {"q": "Show me the top 10 products by revenue", "difficulty": "medium"},
    {"q": "What is the monthly revenue trend?", "difficulty": "medium"},
    {"q": "Which customer segment generates the most revenue?", "difficulty": "medium"},
    {"q": "What is the average order value by region?", "difficulty": "medium"},
    {"q": "What categories have the highest total sales?", "difficulty": "medium"},
    {"q": "Show the return rate by product category", "difficulty": "hard"},
    {"q": "Who are the top 5 customers by lifetime value?", "difficulty": "hard"},
    {"q": "What is the quarter-over-quarter revenue growth?", "difficulty": "hard"},
]

results_log: list[dict] = []

for i, demo in enumerate(DEMO_QUESTIONS, 1):
    print(f"{'='*70}")
    print(f"Demo {i}/10 [{demo['difficulty']}]: {demo['q']}")
    print(f"{'='*70}")

    result = ask_analyst(demo["q"], conn)

    print(f"\n📝 SQL: {result.sql}")
    print(f"✅ Success: {result.success} | Attempts: {result.attempts}")

    if result.assumptions:
        print(f"💡 Assumptions: {', '.join(result.assumptions)}")

    if result.success and result.data is not None:
        print(f"\n📊 Results ({len(result.data)} rows):")
        print(result.data.head(10).to_string(index=False))

        # Narrate
        narration = narrate_results(demo["q"], result.sql, result.data, result.assumptions)
        print(f"\n🗣️ Insight: {narration}")

        # Chart
        chart_type = generate_chart(demo["q"], result.data)
        print(f"📈 Chart type: {chart_type}")
    else:
        print(f"❌ Error: {result.error}")

    results_log.append({
        "question": demo["q"],
        "difficulty": demo["difficulty"],
        "success": result.success,
        "attempts": result.attempts
    })
    print()

In [ ]:
# Summary of demo results

log_df = pd.DataFrame(results_log)
print("Demo results summary:")
print(f"  Total: {len(log_df)}")
print(f"  Success: {log_df['success'].sum()}/{len(log_df)} ({log_df['success'].mean()*100:.0f}%)")
print(f"  Avg attempts: {log_df['attempts'].mean():.1f}")
print(f"\nBy difficulty:")
print(log_df.groupby("difficulty").agg(
    count=("success", "size"),
    success_rate=("success", "mean"),
    avg_attempts=("attempts", "mean")
).to_string())

conn.close()
import os
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"\n🧹 Cleaned up {DB_PATH}")

## 🏋️ Exercises

1. **Complex queries**: Add 5 more questions that require subqueries or window functions
   (e.g., "What is the running total of revenue by month?"). Test them through the full pipeline.

2. **Embedding retrieval**: Replace the keyword-based `retrieve_context` with
   `sentence-transformers` embeddings. Create embeddings for table descriptions and
   column descriptions. Compare retrieval quality on 15 test questions.

3. **Interactive mode**: Build a loop that accepts user input, runs the analyst pipeline,
   and supports follow-up questions (e.g., "drill down into EMEA" after a region query).

## 🎯 Takeaways

- A realistic database with 1000+ rows is essential for testing text-to-SQL systems
- Schema documentation with business context dramatically improves SQL generation quality
- Self-correction with error feedback fixes most failures within 3 attempts
- Narration transforms data tables into actionable business insights
- Auto-chart selection based on question type and data shape saves analyst time
- The full pipeline handles easy, medium, and hard questions with high accuracy

## ⏭️ What's next

In **03_evaluate.ipynb**, we'll rigorously evaluate every component:
SQL correctness, self-correction analysis, narration quality,
schema retrieval precision, and cost/latency benchmarks.

## 📚 References

1. Pourreza, M. & Rafiei, D. (2023). "DIN-SQL: Decomposed In-Context Learning of Text-to-SQL." arXiv:2304.11015
2. Gao, D. et al. (2023). "Text-to-SQL Empowered by Large Language Models: A Benchmark Evaluation." arXiv:2308.15363
3. OpenAI. "GPT-4o mini — API Documentation." https://platform.openai.com/docs
4. Yu, T. et al. (2018). "Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task." EMNLP.